In [42]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

from math import radians, sin, cos, sqrt, atan2

In [43]:
# Navigate to your project
os.chdir('/workspaces/DSE3101-Project')

# Verify you're in the right place
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

# Navigate to data/raw folder
os.chdir('data/raw')

Current directory: /workspaces/DSE3101-Project
Contents: ['docs', '.git', 'notebooks', '.DS_Store', 'models', 'frontend', 'requirements.txt', '.gitignore', '.vscode', 'backend', 'requirements_fixed.txt', 'Final Web Scraper.ipynb', 'data', 'onemap_all_themes_full.json', 'README.md', 'onemap_all_themes_raw.txt']


In [44]:
df = pd.read_csv('propertyguru_final.csv')

# Check to see if it loaded correctly
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20883 entries, 0 to 20882
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        20883 non-null  str    
 1   exact_address                      20880 non-null  str    
 2   price_detail                       20702 non-null  str    
 3   town_detail                        20800 non-null  str    
 4   bedrooms_detail                    20800 non-null  float64
 5   bathrooms_detail                   20161 non-null  float64
 6   area_detail                        20701 non-null  str    
 7   floor_detail                       9935 non-null   str    
 8   property_type_detail               20800 non-null  str    
 9   address_from_url                   20883 non-null  str    
 10  listing_id                         20883 non-null  int64  
 11  postal_code                        20265 non-null  float64
 12  o

In [45]:
# Count how many rows are duplicate (i.e., not the first occurrence)
num_duplicates = df.duplicated(subset=["listing_url"]).sum()
print("Number of duplicate rows:", num_duplicates)

df = df.drop_duplicates(subset=["listing_url"], keep="first")

Number of duplicate rows: 6112


In [46]:
df.shape

(14771, 33)

In [47]:
# Missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

Missing values per column:
listing_url                              0
exact_address                            2
price_detail                           128
town_detail                             61
bedrooms_detail                         61
bathrooms_detail                       515
area_detail                            129
floor_detail                          7773
property_type_detail                    61
address_from_url                         0
listing_id                               0
postal_code                            616
onemap_full_address                    608
onemap_road_name                       608
latitude                               608
longitude                              608
nearest_mrt_name                       608
nearest_mrt_exit                       608
nearest_mrt_distance_m                 608
num_mrt_within_1000m                   608
nearest_clinic_name                  14623
nearest_clinic_distance_m              608
nearest_park_name          

In [48]:
critical_cols = [
    'price_detail', # important to get over/under valuation label
    'area_detail', # important to predict fair valuation price using xgboost model
    'num_amenities_within_1000m', # important to calculate SAI score
]

# Drop rows where any of the critical columns have missing values
clean_df = df.dropna(subset=critical_cols)

In [49]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 14042 entries, 0 to 20882
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        14042 non-null  str    
 1   exact_address                      14042 non-null  str    
 2   price_detail                       14042 non-null  str    
 3   town_detail                        14042 non-null  str    
 4   bedrooms_detail                    14042 non-null  float64
 5   bathrooms_detail                   13664 non-null  float64
 6   area_detail                        14042 non-null  str    
 7   floor_detail                       6877 non-null   str    
 8   property_type_detail               14042 non-null  str    
 9   address_from_url                   14042 non-null  str    
 10  listing_id                         14042 non-null  int64  
 11  postal_code                        14034 non-null  float64
 12  onemap

In [50]:
clean_df['room_count'] = clean_df['bedrooms_detail'] + 1 # Living room

In [51]:
clean_df['room_count'].value_counts()

room_count
4.0     9155
3.0     2844
5.0     1753
2.0      191
6.0       76
7.0       20
1.0        2
32.0       1
Name: count, dtype: int64

In [52]:
clean_df["floor_area_sqm"] = clean_df["area_detail"].str.replace(r"\D+", "", regex=True).astype(float) * 0.092903

In [53]:
# Filter for listings with rooms <= 4 (relevant for downsizing)
mask_bedrooms = clean_df["room_count"] <= 4
mask_hdb = clean_df["listing_url"].str.contains("hdb", na=False, case=False)

final_df = clean_df[mask_bedrooms & mask_hdb]

In [54]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 12192 entries, 0 to 20882
Data columns (total 35 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        12192 non-null  str    
 1   exact_address                      12192 non-null  str    
 2   price_detail                       12192 non-null  str    
 3   town_detail                        12192 non-null  str    
 4   bedrooms_detail                    12192 non-null  float64
 5   bathrooms_detail                   11823 non-null  float64
 6   area_detail                        12192 non-null  str    
 7   floor_detail                       6082 non-null   str    
 8   property_type_detail               12192 non-null  str    
 9   address_from_url                   12192 non-null  str    
 10  listing_id                         12192 non-null  int64  
 11  postal_code                        12185 non-null  float64
 12  onemap

In [55]:
# HDB Towns in Singapore:
# https://www.propertyguru.com.sg/singapore-property-listing/hdb/hdb-singapore-estates# 

station_to_town = {
    'ADMIRALTY MRT STATION': 'Woodlands',
    'ALJUNIED MRT STATION': 'Geylang',
    'ANG MO KIO MRT STATION': 'Ang Mo Kio',
    'BAKAU LRT STATION': 'Sengkang',
    'BANGKIT LRT STATION': 'Bukit Panjang',
    'BARTLEY MRT STATION': 'Serangoon',
    'BAYSHORE MRT STATION': 'Bedok',
    'BEAUTY WORLD MRT STATION': 'Bukit Timah',
    'BEDOK MRT STATION': 'Bedok',
    'BEDOK NORTH MRT STATION': 'Bedok',
    'BEDOK RESERVOIR MRT STATION': 'Bedok',
    'BENCOOLEN MRT STATION': 'Central Area',
    'BENDEMEER MRT STATION': 'Kallang/Whampoa',
    'BISHAN MRT STATION': 'Bishan',
    'BOON KENG MRT STATION': 'Kallang/Whampoa',
    'BOON LAY MRT STATION': 'Jurong West',
    'BRADDELL MRT STATION': 'Toa Payoh',
    'BRIGHT HILL MRT STATION': 'Bishan',
    'BUANGKOK MRT STATION': 'Sengkang',
    'BUGIS MRT STATION': 'Central Area',
    'BUKIT BATOK MRT STATION': 'Bukit Batok',
    'BUKIT GOMBAK MRT STATION': 'Bukit Batok',
    'BUKIT PANJANG LRT STATION': 'Bukit Panjang',
    'BUKIT PANJANG MRT STATION': 'Bukit Panjang',
    'BUONA VISTA MRT STATION': 'Queenstown',
    'CALDECOTT MRT STATION': 'Toa Payoh',
    'CANBERRA MRT STATION': 'Sembawang',
    'CC9': 'Central Area',
    'CHANGI AIRPORT MRT STATION': 'Tampines',
    'CHENG LIM LRT STATION': 'Sengkang',
    'CHINATOWN MRT STATION': 'Central Area',
    'CHINESE GARDEN MRT STATION': 'Jurong East',
    'CHOA CHU KANG MRT STATION': 'Choa Chu Kang',
    'CLEMENTI MRT STATION': 'Clementi',
    'COMMONWEALTH MRT STATION': 'Queenstown',
    'COMPASSVALE LRT STATION': 'Sengkang',
    'CORAL EDGE LRT STATION': 'Punggol',
    'COVE LRT STATION': 'Punggol',
    'DAKOTA MRT STATION': 'Geylang',
    'DAMAI LRT STATION': 'Punggol',
    'DOVER MRT STATION': 'Queenstown',
    'DT4': 'Central Area',
    'EUNOS MRT STATION': 'Geylang',
    'FAJAR LRT STATION': 'Bukit Panjang',
    'FARRER PARK MRT STATION': 'Kallang/Whampoa',
    'FARRER ROAD MRT STATION': 'Bukit Timah',
    'FARMWAY LRT STATION': 'Sengkang',
    'FERNVALE LRT STATION': 'Sengkang',
    'GEYLANG BAHRU MRT STATION': 'Kallang/Whampoa',
    'GREAT WORLD MRT STATION': 'Central Area',
    'HARBOURFRONT MRT STATION': 'Bukit Merah',
    'HAVELOCK MRT STATION': 'Bukit Merah',
    'HOLLAND VILLAGE MRT STATION': 'Queenstown',
    'HOUGANG MRT STATION': 'Hougang',
    'JALAN BESAR MRT STATION': 'Central Area',
    'JELAPANG LRT STATION': 'Bukit Panjang',
    'JURONG EAST MRT STATION': 'Jurong East',
    'KADALOOR LRT STATION': 'Punggol',
    'KAKI BUKIT MRT STATION': 'Bedok',
    'KALLANG MRT STATION': 'Kallang/Whampoa',
    'KANGKAR LRT STATION': 'Sengkang',
    'KATONG PARK MRT STATION': 'Marine Parade',
    'KEAT HONG LRT STATION': 'Choa Chu Kang',
    'KEMBANGAN MRT STATION': 'Bedok',
    'KHATIB MRT STATION': 'Yishun',
    'KOVAN MRT STATION': 'Hougang',
    'KUPANG LRT STATION': 'Sengkang',
    'LABRADOR PARK MRT STATION': 'Bukit Merah',
    'LAKESIDE MRT STATION': 'Jurong West',
    'LAVENDER MRT STATION': 'Kallang/Whampoa',
    'LAYAR LRT STATION': 'Sengkang',
    'LENTOR MRT STATION': 'Ang Mo Kio',
    'LITTLE INDIA MRT STATION': 'Central Area',
    'LORONG CHUAN MRT STATION': 'Serangoon',
    'MACPHERSON MRT STATION': 'Geylang',
    'MARINE PARADE MRT STATION': 'Marine Parade',
    'MARINE TERRACE MRT STATION': 'Marine Parade',
    'MARSILING MRT STATION': 'Woodlands',
    'MARYMOUNT MRT STATION': 'Bishan',
    'MATTAR MRT STATION': 'Geylang',
    'MAXWELL MRT STATION': 'Central Area',
    'MAYFLOWER MRT STATION': 'Ang Mo Kio',
    'MERIDIAN LRT STATION': 'Punggol',
    'MOUNTBATTEN MRT STATION': 'Geylang',
    'NIBONG LRT STATION': 'Punggol',
    'NICOLL HIGHWAY MRT STATION': 'Central Area',
    'NOVENA MRT STATION': 'Kallang/Whampoa',
    'OASIS LRT STATION': 'Punggol',
    'ONE-NORTH MRT STATION': 'Queenstown',
    'OUTRAM PARK MRT STATION': 'Central Area',
    'PASIR RIS MRT STATION': 'Pasir Ris',
    'PAYA LEBAR MRT STATION': 'Geylang',
    'PENDING LRT STATION': 'Bukit Panjang',
    'PETIR LRT STATION': 'Bukit Panjang',
    'PHOENIX LRT STATION': 'Bukit Panjang',
    'PIONEER MRT STATION': 'Jurong West',
    'POTONG PASIR MRT STATION': 'Toa Payoh',
    'PUNGGOL LRT STATION': 'Punggol',
    'PUNGGOL MRT STATION': 'Punggol',
    'PUNGGOL POINT LRT STATION': 'Punggol',
    'QUEENSTOWN MRT STATION': 'Queenstown',
    'RANGGUNG LRT STATION': 'Sengkang',
    'REDHILL MRT STATION': 'Bukit Merah',
    'RENJONG LRT STATION': 'Sengkang',
    'RIVIERA LRT STATION': 'Punggol',
    'ROCHOR MRT STATION': 'Central Area',
    'RUMBIA LRT STATION': 'Sengkang',
    'SAMUDERA LRT STATION': 'Punggol',
    'SEGAR LRT STATION': 'Bukit Panjang',
    'SEMBAWANG MRT STATION': 'Sembawang',
    'SENGKANG MRT STATION': 'Sengkang',
    'SENJA LRT STATION': 'Bukit Panjang',
    'SERANGOON MRT STATION': 'Serangoon',
    'SIMEI MRT STATION': 'Tampines',
    'SOO TECK LRT STATION': 'Punggol',
    'SOUTH VIEW LRT STATION': 'Choa Chu Kang',
    'SUMANG LRT STATION': 'Punggol',
    'TAI SENG MRT STATION': 'Geylang',
    'TAMPINES EAST MRT STATION': 'Tampines',
    'TAMPINES MRT STATION': 'Tampines',
    'TAMPINES WEST MRT STATION': 'Tampines',
    'TANAH MERAH MRT STATION': 'Bedok',
    'TANJONG PAGAR MRT STATION': 'Central Area',
    'TECK WHYE LRT STATION': 'Choa Chu Kang',
    'TELOK BLANGAH MRT STATION': 'Bukit Merah',
    'THANGGAM LRT STATION': 'Sengkang',
    'TIONG BAHRU MRT STATION': 'Bukit Merah',
    'TOA PAYOH MRT STATION': 'Toa Payoh',
    'TONGKANG LRT STATION': 'Sengkang',
    'UBI MRT STATION': 'Geylang',
    'UPPER CHANGI MRT STATION': 'Tampines',
    'UPPER THOMSON MRT STATION': 'Bishan',
    'WOODLANDS MRT STATION': 'Woodlands',
    'WOODLANDS NORTH MRT STATION': 'Woodlands',
    'WOODLANDS SOUTH MRT STATION': 'Woodlands',
    'WOODLEIGH MRT STATION': 'Toa Payoh',
    'YEW TEE MRT STATION': 'Choa Chu Kang',
    'YIO CHU KANG MRT STATION': 'Ang Mo Kio',
    'YISHUN MRT STATION': 'Yishun'
}

In [56]:
final_df = final_df.dropna(subset=['nearest_mrt_name'])
final_df['hdb_town'] = final_df['nearest_mrt_name'].map(station_to_town)

# To check if any stations were missed (print rows where the mapping failed)
missing_towns = final_df[final_df['hdb_town'].isna()]
if not missing_towns.empty:
    print("Warning: Some stations were not mapped!")
    print(missing_towns['nearest_mrt_name'].unique())
else:
    print("All stations successfully mapped to an HDB town!")

All stations successfully mapped to an HDB town!


In [57]:
final_df['hdb_town'].value_counts()

hdb_town
Sengkang           1235
Punggol             911
Tampines            814
Yishun              795
Jurong West         698
Bukit Batok         682
Woodlands           608
Ang Mo Kio          593
Bedok               585
Toa Payoh           581
Bukit Merah         518
Queenstown          500
Hougang             485
Choa Chu Kang       398
Kallang/Whampoa     374
Bukit Panjang       364
Clementi            310
Sembawang           308
Geylang             290
Bishan              241
Jurong East         234
Pasir Ris           225
Central Area        167
Serangoon           149
Marine Parade        89
Bukit Timah          38
Name: count, dtype: int64

In [58]:
region_mapping = {
    # --- CCR (Core Central Region) ---
    'ORCHARD': 'CCR', 'SOMERSET': 'CCR', 'RIVER VALLEY': 'CCR',
    'TANGLIN': 'CCR', 'BUKIT TIMAH': 'CCR', 'HOLLAND': 'CCR',
    'NEWTON': 'CCR', 'NOVENA': 'CCR', 'DUNEARN': 'CCR', 'WATTEN': 'CCR',
    'BOAT QUAY': 'CCR', 'RAFFLES PLACE': 'CCR', 'MARINA DOWNTOWN': 'CCR', 'SUNTEC CITY': 'CCR',
    'SHENTON WAY': 'CCR', 'TANJONG PAGAR': 'CCR',
    'SENTOSA': 'CCR',
    'CITY HALL': 'CCR',
    'BUGIS': 'CCR',
    'CENTRAL AREA': 'CCR', # Official HDB Town

    # --- RCR (Rest of Central Region) ---
    'MARINA SOUTH': 'RCR',
    'CHINATOWN': 'RCR',
    'QUEENSTOWN': 'RCR', 'ALEXANDRA': 'RCR', 'TIONG BAHRU': 'RCR',
    'HARBOURFRONT': 'RCR', 'KEPPEL': 'RCR', 'TELOK BLANGAH': 'RCR',
    'BUONA VISTA': 'RCR', 'DOVER': 'RCR', 'PASIR PANJANG': 'RCR',
    'FORT CANNING': 'RCR',
    'ROCHOR': 'RCR',
    'LITTLE INDIA': 'RCR', 'FARRER PARK': 'RCR',
    'BALESTIER': 'RCR', 'WHAMPOA': 'RCR', 'TOA PAYOH': 'RCR', 'BOON KENG': 'RCR', 'BENDEMEER': 'RCR', 'KAMPONG BUGIS': 'RCR',
    'POTONG PASIR': 'RCR', 'BIDADARI': 'RCR', 'MACPHERSON': 'RCR', 'UPPER ALJUNIED': 'RCR',
    'GEYLANG': 'RCR', 'DAKOTA': 'RCR', 'PAYA LEBAR CENTRAL': 'RCR', 'EUNOS': 'RCR', 'UBI': 'RCR', 'ALJUNIED': 'RCR',
    'TANJONG RHU': 'RCR', 'AMBER': 'RCR', 'MEYER': 'RCR', 'KATONG': 'RCR', 'DUNMAN': 'RCR', 'JOO CHIAT': 'RCR', 'MARINE PARADE': 'RCR',
    'BISHAN': 'RCR', 'THOMSON': 'RCR',
    'BUKIT MERAH': 'RCR',      # Official HDB Town
    'KALLANG/WHAMPOA': 'RCR',  # Official HDB Town

    # --- OCR (Outside Central Region) ---
    'CLEMENTI': 'OCR', 'WEST COAST': 'OCR',
    'KEMBANGAN': 'OCR', 'KAKI BUKIT': 'OCR',
    'TELOK KURAU': 'OCR', 'SIGLAP': 'OCR', 'FRANKEL': 'OCR',
    'BEDOK': 'OCR', 'UPPER EAST COAST': 'OCR', 'BAYSHORE': 'OCR', 'TANAH MERAH': 'OCR', 'UPPER CHANGI': 'OCR',
    'FLORA DRIVE': 'OCR', 'LOYANG': 'OCR', 'CHANGI': 'OCR',
    'TAMPINES': 'OCR', 'PASIR RIS': 'OCR',
    'PUNGGOL': 'OCR', 'SENGKANG': 'OCR', 'HOUGANG': 'OCR', 'KOVAN': 'OCR', 'SERANGOON': 'OCR', 'LORONG AH SOO': 'OCR',
    'ANG MO KIO': 'OCR',
    'UPPER BUKIT TIMAH': 'OCR', 'ULU PANDAN': 'OCR', 'CLEMENTI PARK': 'OCR',
    'JURONG EAST': 'OCR', 'JURONG WEST': 'OCR', 'BOON LAY': 'OCR',
    'HILLVIEW': 'OCR', 'BUKIT PANJANG': 'OCR', 'BUKIT BATOK': 'OCR', 'CHOA CHU KANG': 'OCR',
    'KRANJI': 'OCR', 'LIM CHU KANG': 'OCR', 'SUNGEI GEDONG': 'OCR', 'TENGAH': 'OCR',
    'WOODLANDS': 'OCR', 'ADMIRALTY': 'OCR',
    'LENTOR': 'OCR', 'SPRINGLEAF': 'OCR', 'MANDAI': 'OCR',
    'YISHUN': 'OCR', 'SEMBAWANG': 'OCR',
    'SELETAR': 'OCR', 'SELETAR HILL': 'OCR', 'SENGKANG WEST': 'OCR'
}

# Apply the mapping. 
# Using .str.upper() ensures that even if your dataframe has 'Orchard' or 'orchard', it will match the uppercase keys.
final_df['region'] = final_df['hdb_town'].str.upper().map(region_mapping)
final_df['region'] = final_df['region'].fillna('Other')
print("✓ Created region feature")

# Flag mature vs non-mature estates
mature_estates = ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT MERAH', 'BUKIT TIMAH',
                  'CENTRAL AREA', 'CLEMENTI', 'GEYLANG', 'KALLANG/WHAMPOA',
                  'MARINE PARADE', 'PASIR RIS', 'QUEENSTOWN', 'SERANGOON', 'TAMPINES', 'TOA PAYOH']

final_df['is_mature_estate'] = final_df['hdb_town'].isin(mature_estates).astype(int)
print("✓ Created is_mature_estate flag")

✓ Created region feature
✓ Created is_mature_estate flag


In [59]:
# Remove duplicate rows if any
print(final_df.duplicated().sum())
final_df = final_df.drop_duplicates()

0


In [60]:
final_df.columns.tolist()

['listing_url',
 'exact_address',
 'price_detail',
 'town_detail',
 'bedrooms_detail',
 'bathrooms_detail',
 'area_detail',
 'floor_detail',
 'property_type_detail',
 'address_from_url',
 'listing_id',
 'postal_code',
 'onemap_full_address',
 'onemap_road_name',
 'latitude',
 'longitude',
 'nearest_mrt_name',
 'nearest_mrt_exit',
 'nearest_mrt_distance_m',
 'num_mrt_within_1000m',
 'nearest_clinic_name',
 'nearest_clinic_distance_m',
 'nearest_park_name',
 'nearest_park_distance_m',
 'nearest_community_club_name',
 'nearest_community_club_distance_m',
 'nearest_hawker_name',
 'nearest_hawker_distance_m',
 'num_clinic_within_1000m',
 'num_park_within_1000m',
 'num_community_club_within_1000m',
 'num_hawker_within_1000m',
 'num_amenities_within_1000m',
 'room_count',
 'floor_area_sqm',
 'hdb_town',
 'region',
 'is_mature_estate']

In [61]:
# See all values in a clean readable format
for value, count in final_df['floor_detail'].value_counts().items():
    print(f"{count:>6} | {value}")

  2122 | High Floor
  1399 | High floor
   727 | high floor
   329 | Mid floor
   324 | Mid Floor
   264 | Low floor
   171 | Low Floor
   122 | low floor
   115 | mid floor
    65 | HIGH FLOOR
    35 | 50th floor
    30 | a high floor
    21 | 2nd floor
    21 | MID FLOOR
    13 | 10th floor
    10 | 2nd Floor
    10 | LOW FLOOR
    10 | 20th floor
     9 | 6th floor
     9 | 4th Floor
     7 | 8th Floor
     6 | a low floor
     6 | 5th floor
     6 | 7th floor
     6 | 3rd floor
     6 | 4th floor
     5 | 3rd Floor
     4 | 15th floor
     4 | 10th Floor
     4 | 8th floor
     4 | 1st floor
     4 | 12th Floor
     4 | 6th Floor
     4 | 11th floor
     4 | a mid floor
     4 | High mid-floor position for privacy and elevated views
     3 | high level
     3 | high Floor
     3 | HIGH floor
     3 | 30th floor
     3 | a Super high floor
     3 | HDB flat type
     2 | 12th floor
     2 | 11th Floor
     2 | a high floor and North-South facing
     2 | 14th Floor
     2 | 11TH FLO

In [62]:
import re
import numpy as np
import pandas as pd

# ============================================================
# STANDARDISED FLOOR RANGES (consistent throughout)
# Low:  Floor 1  - 5
# Mid:  Floor 6  - 12
# High: Floor 13+
# ============================================================

def map_floor_category(text):
    if pd.isna(text) or str(text).strip() == '':
        return np.nan

    text_clean = str(text).lower().strip()

    # ============================================================
    # RULE 1: Extract explicit floor numbers
    # Handles: "10th floor", "2nd floor", "Level 11", "09 floor"
    # ============================================================
    # Handle "level X" format
    level_match = re.search(r'level\s*(\d+)', text_clean)
    if level_match:
        floor_num = int(level_match.group(1))
        return num_to_category(floor_num)

    # Handle "Xth/Xnd/Xrd/Xst floor" format
    number_match = re.search(r'\b(\d+)\s*(?:st|nd|rd|th)?\s*floor', text_clean)
    if number_match:
        floor_num = int(number_match.group(1))
        return num_to_category(floor_num)

    # Handle "floor X" format (e.g. "floor 9")
    floor_num_match = re.search(r'floor\s*(\d+)', text_clean)
    if floor_num_match:
        floor_num = int(floor_num_match.group(1))
        return num_to_category(floor_num)

    # ============================================================
    # RULE 2: Handle "above X" phrases
    # Handles: "above level 6", "above 10", "12 and above"
    # ============================================================
    above_match = re.search(r'above\s*(?:level|the)?\s*(\d+)', text_clean)
    if above_match:
        floor_num = int(above_match.group(1))
        return num_to_category(floor_num + 1)  # "above 6" = floor 7

    and_above_match = re.search(r'(\d+)\s*and\s*above', text_clean)
    if and_above_match:
        floor_num = int(and_above_match.group(1))
        return num_to_category(floor_num)

    # ============================================================
    # RULE 3: Standalone numbers only
    # Handles: "15", "32", "4", "17"
    # Must come BEFORE keyword rules to avoid matching
    # numbers inside garbage text like "3 bedrooms"
    # ============================================================
    standalone_match = re.fullmatch(r'\s*(\d+)\s*', text_clean)
    if standalone_match:
        floor_num = int(standalone_match.group(1))
        return num_to_category(floor_num)

    # ============================================================
    # RULE 4: Keyword combinations
    # Order matters: most specific phrases checked first
    # ============================================================

    # Super / very high → high
    if any(phrase in text_clean for phrase in [
        'super high', 'very high', 'highest', 'top floor', 'top level',
        'a very high', 'a super high'
    ]):
        return 'high'

    # Mid to high / mid-high → high (lean towards high)
    if any(phrase in text_clean for phrase in [
        'mid to high', 'mid-to-high', 'mid-high', 'mid high'
    ]):
        return 'high'

    # High (all variations)
    if any(phrase in text_clean for phrase in [
        'high floor', 'high level', 'higher floor',
        'high unit', 'high uni', ' high '
    ]):
        return 'high'
    if text_clean in ['high']:
        return 'high'

    # Mid (all variations)
    if any(phrase in text_clean for phrase in [
        'mid floor', 'mid level', 'mid-floor', 'mid-level',
        'middle floor', 'mid unit', 'mid '
    ]):
        return 'mid'
    if text_clean in ['mid']:
        return 'mid'

    # Low (all variations)
    if any(phrase in text_clean for phrase in [
        'low floor', 'low level', 'lower floor', 'low unit',
        'ground floor', '1st floor', 'corridor level', 'low '
    ]):
        return 'low'
    if text_clean in ['low']:
        return 'low'

    # ============================================================
    # RULE 5: Catch remaining directional keywords
    # Handles: "a higher floor", "the lower floor"
    # ============================================================
    if 'lower' in text_clean:
        return 'low'
    if 'higher' in text_clean:
        return 'high'

    # ============================================================
    # RULE 6: Truly unclassifiable garbage values
    # Handles: "Bendemeer Road", "3 bedrooms", "Canberra Street"
    # Return NaN and let the fallback fill them in
    # ============================================================
    return np.nan


def num_to_category(floor_num):
    """
    Standardised floor number to category mapping.
    Consistent ranges used throughout the entire function.
    Low:  1  - 5
    Mid:  6  - 12
    High: 13+
    """
    if floor_num <= 5:
        return 'low'
    elif floor_num <= 12:
        return 'mid'
    else:
        return 'high'


# ============================================================
# APPLY AND HANDLE ALL NaN VALUES
# ============================================================

# Step 1: Apply the mapping function
final_df['floor_category'] = final_df['floor_detail'].apply(map_floor_category)

print("Before NaN handling:")
print(final_df['floor_category'].value_counts(dropna=False))
print(f"NaN count: {final_df['floor_category'].isna().sum()}")

# Step 2: Map floor category to numeric storey_mid
floor_to_storey = {
    'low':  3,   # Midpoint of floors 1-5
    'mid':  9,   # Midpoint of floors 6-12
    'high': 16   # Representative of floors 13+
}
final_df['storey_mid'] = final_df['floor_category'].map(floor_to_storey)

# Step 3: Fill remaining NaN with town-level median (broader fallback)
final_df['storey_mid'] = final_df.groupby(
    'hdb_town')['storey_mid'].transform(
    lambda x: x.fillna(x.median())
)

# Step 4: Fill any last remaining NaN with global median (final fallback)
global_median = final_df['storey_mid'].median()
final_df['storey_mid'] = final_df['storey_mid'].fillna(global_median)

# Step 5: Backfill floor_category from storey_mid for NaN rows
# So floor_category is also always filled
def storey_mid_to_category(mid):
    if mid <= 5:
        return 'low'
    elif mid <= 12:
        return 'mid'
    else:
        return 'high'

final_df['floor_category'] = final_df.apply(
    lambda row: storey_mid_to_category(row['storey_mid'])
    if pd.isna(row['floor_category'])
    else row['floor_category'],
    axis=1
)

print("\nAfter NaN handling:")
print(final_df['floor_category'].value_counts(dropna=False))
print(f"NaN count: {final_df['floor_category'].isna().sum()}")
print(f"Global median used as final fallback: {global_median}")

Before NaN handling:
floor_category
NaN     6156
high    4484
mid      884
low      668
Name: count, dtype: int64
NaN count: 6156

After NaN handling:
floor_category
high    10618
mid       906
low       668
Name: count, dtype: int64
NaN count: 0
Global median used as final fallback: 16.0


In [63]:
def map_flat_type(room_count):
    room_map = {
        2: '2 ROOM',
        3: '3 ROOM',
        4: '4 ROOM',
        5: '5 ROOM',
        6: 'EXECUTIVE'
    }
    
    if pd.isna(room_count):
        return '4 ROOM'  # Most common HDB flat type as fallback
    
    return room_map.get(int(room_count), '4 ROOM')  # Default to 4 ROOM if unknown

final_df['flat_type'] = final_df['room_count'].apply(map_flat_type)

print("Flat type created:")
print(final_df['flat_type'].value_counts())
print(f"\nNaN count: {final_df['flat_type'].isna().sum()}")

Flat type created:
flat_type
4 ROOM    9157
3 ROOM    2844
2 ROOM     191
Name: count, dtype: int64

NaN count: 0


In [64]:
import pandas as pd
import re

def extract_block(text: str) -> str:
    match = re.search(r"^\D*(\d+[A-Za-z]?)", text.strip()) if pd.notna(text) else None
    return match.group(1) if match else ""

final_df["block_number"] = final_df["onemap_full_address"].apply(extract_block)
final_df["standardised_address"] = final_df["block_number"] + " " + final_df["onemap_road_name"].fillna("")

In [65]:
final_df['standardised_address'].value_counts().head(25)

standardised_address
104A BIDADARI PARK DRIVE          39
118A ALKAFF CRESCENT              39
1 CANTONMENT ROAD                 38
94 DAWSON ROAD                    28
467B BUKIT BATOK WEST AVENUE 9    23
103B BIDADARI PARK DRIVE          20
422A NORTHSHORE DRIVE             20
88 DAWSON ROAD                    19
101A BIDADARI PARK DRIVE          19
228A ANG MO KIO STREET 23         19
96 DAWSON ROAD                    18
464A BUKIT BATOK WEST AVENUE 8    18
409D NORTHSHORE DRIVE             17
228 TAMPINES STREET 23            17
227A ANG MO KIO STREET 23         16
86 DAWSON ROAD                    16
95 DAWSON ROAD                    16
103A BIDADARI PARK DRIVE          15
453B BUKIT BATOK WEST AVENUE 6    15
93B TELOK BLANGAH STREET 31       15
90A TELOK BLANGAH STREET 31       15
452A BUKIT BATOK WEST AVENUE 6    15
619A TAMPINES STREET 61           14
618A TAMPINES STREET 61           14
775 YISHUN RING ROAD              14
Name: count, dtype: int64

In [66]:
# ─────────────────────────────────────────────────────────────────
# 2. NORMALISE
#    Fixes encoding, strips "Blk/Block", expands abbreviations,
#    title-cases, and collapses whitespace.
# ─────────────────────────────────────────────────────────────────
_ABBREV_MAP = [
    # Road types
    (r"\bAve\b",    "Avenue"),
    (r"\bAv\b",     "Avenue"),
    (r"\bSt\b",     "Street"),
    (r"\bRd\b",     "Road"),
    (r"\bDr\b",     "Drive"),
    (r"\bCres\b",   "Crescent"),
    (r"\bCrst\b",   "Crescent"),
    (r"\bCrt\b",    "Crescent"),
    (r"\bCl\b",     "Close"),
    (r"\bCls\b",    "Close"),
    (r"\bLn\b",     "Lane"),
    (r"\bTce\b",    "Terrace"),
    (r"\bTer\b",    "Terrace"),
    (r"\bBvd\b",    "Boulevard"),
    (r"\bBlvd\b",   "Boulevard"),
    (r"\bWk\b",     "Walk"),
    (r"\bWy\b",     "Way"),
    (r"\bLk\b",     "Link"),
    (r"\bHwy\b",    "Highway"),
    (r"\bRise\b",   "Rise"),       # already full but kept for consistency

    # Directional
    (r"\bNth\b",    "North"),
    (r"\bNth\.\b",  "North"),
    (r"\bSth\b",    "South"),
    (r"\bSth\.\b",  "South"),
    (r"\bEst\b",    "East"),
    (r"\bWst\b",    "West"),
    (r"\bNE\b",     "North East"),
    (r"\bNW\b",     "North West"),
    (r"\bSE\b",     "South East"),
    (r"\bSW\b",     "South West"),

    # Common words
    (r"\bCtrl\b",   "Central"),
    (r"\bCtr\b",    "Central"),
    (r"\bCent\b",   "Central"),
    (r"\bCentral\b","Central"),    # already full
    (r"\bJln\b",    "Jalan"),
    (r"\bJl\b",     "Jalan"),
    (r"\bLor\b",    "Lorong"),
    (r"\bLrg\b",    "Lorong"),
    (r"\bHts\b",    "Heights"),
    (r"\bHt\b",     "Heights"),
    (r"\bPl\b",     "Place"),
    (r"\bGdn\b",    "Garden"),
    (r"\bGdns\b",   "Gardens"),
    (r"\bVw\b",     "View"),
    (r"\bPk\b",     "Park"),
    (r"\bPrk\b",    "Park"),
    (r"\bGrn\b",    "Green"),
    (r"\bGr\b",     "Green"),
    (r"\bGlen\b",   "Glen"),       # already full
    (r"\bRes\b",    "Reservoir"),
    (r"\bRsvr\b",   "Reservoir"),
    (r"\bMkt\b",    "Market"),
    (r"\bStn\b",    "Station"),
    (r"\bHse\b",    "House"),
    (r"\bBldg\b",   "Building"),
    (r"\bApt\b",    "Apartment"),
    (r"\bApts\b",   "Apartments"),

    # Singapore-specific area prefixes/words
    (r"\bBt\b",     "Bukit"),
    (r"\bBkt\b",    "Bukit"),
    (r"\bBk\b",     "Bukit"),
    (r"\bTg\b",     "Tanjong"),
    (r"\bTjg\b",    "Tanjong"),
    (r"\bTanj\b",   "Tanjong"),
    (r"\bKg\b",     "Kampong"),
    (r"\bKpg\b",    "Kampong"),
    (r"\bKamp\b",   "Kampong"),
    (r"\bPsr\b",    "Pasir"),
    (r"\bPg\b",     "Punggol"),
    (r"\bAMK\b",    "Ang Mo Kio"),
    (r"\bBBK\b",    "Bukit Batok"),
    (r"\bBP\b",     "Bukit Panjang"),
    (r"\bBM\b",     "Bukit Merah"),
    (r"\bBT\b",     "Bukit Timah"),
    (r"\bCCK\b",    "Choa Chu Kang"),
    (r"\bHG\b",     "Hougang"),
    (r"\bJE\b",     "Jurong East"),
    (r"\bJW\b",     "Jurong West"),
    (r"\bTB\b",     "Telok Blangah"),
    (r"\bWL\b",     "Woodlands"),
    (r"\bYS\b",     "Yishun"),
    (r"\bSKG\b",    "Sengkang"),
    (r"\bSKW\b",    "Sengkang West"),
    (r"\bTPY\b",    "Toa Payoh"),
    (r"\bSRN\b",    "Serangoon"),
    (r"\bTPS\b",    "Tampines"),
    (r"\bPR\b",     "Pasir Ris"),
    (r"\bBDK\b",    "Bedok"),
    (r"\bCLM\b",    "Clementi"),
]

def normalise(addr: str) -> str:
    s = addr.strip()
    if not s:
        return ""

    # Fix mojibake (Â + non-breaking space from latin1/UTF-8 mismatch)
    s = s.replace("Â\xa0", " ").replace("Â ", " ").replace("\xa0", " ").replace("Â", "")

    # Strip leading Block / Blk (with or without space before number, e.g. "Blk301")
    s = re.sub(r"^(Block|Blk)\s*", "", s, flags=re.IGNORECASE)

    # Remove dots used as separators between letter and digit (e.g. "St.31" → "St 31")
    s = re.sub(r"(?<=[A-Za-z])\.(?=\d)", " ", s)

    # Strip trailing marketing noise after "!" (keep the address part)
    s = re.sub(r"\s*!\s.*$", "", s).strip()

    # Expand abbreviations
    for pattern, replacement in _ABBREV_MAP:
        s = re.sub(pattern, replacement, s, flags=re.IGNORECASE)

    # Title-case
    s = s.title()

    # Restore lowercase for common prepositions broken by title-case
    for word in ("the", "of", "at", "to", "and", "a"):
        s = re.sub(r"\b" + word.title() + r"\b", word, s)

    # Collapse whitespace
    return re.sub(r"\s{2,}", " ", s).strip()

In [67]:
# Apply normalise to both columns
final_df["address_norm"] = final_df["standardised_address"].apply(lambda x: normalise(str(x)) if pd.notna(x) else "")

# Download from: https://data.gov.sg/dataset/hdb-property-information
hdb_info_df = pd.read_csv('HDBPropertyInformation.csv')

# Create a matching address key in the HDB dataset
# Combines block + street e.g. "123 ANG MO KIO AVE 3"
hdb_info_df['address'] = (
    hdb_info_df['blk_no'].astype(str).str.strip() + " " +
    hdb_info_df['street'].astype(str).str.strip()
)
hdb_info_df['address'] = hdb_info_df['address'].str.upper().str.strip()

hdb_info_df["address_norm"] = hdb_info_df["address"].apply(lambda x: normalise(str(x)) if pd.notna(x) else "")

In [68]:
final_df['address_norm'].value_counts().sample(10)

address_norm
122 Bedok North Street 2       1
138C Yuan Ching Road           7
127 Tampines Street 11         2
440A Clementi Avenue 3         3
93A Telok Blangah Street 31    9
619 Woodlands Drive 52         2
428B Yishun Avenue 11          4
242 Kim Keat Link              2
34 Circuit Road                2
152 Jalan Teck Whye            3
Name: count, dtype: int64

In [69]:
hdb_info_df['address_norm'].value_counts().sample(10)

address_norm
81 Tiong Poh Road             1
663A Jurong West Street 65    1
804 Woodlands Street 81       1
447 Choa Chu Kang Avenue 4    1
427 Bedok North Road          1
268 Boon Lay Drive            1
503B Canberra Link            1
678 Hougang Avenue 8          1
683A Jurong West Central 1    1
555 Jurong West Street 42     1
Name: count, dtype: int64

In [70]:
# ============================================================
# LEASE FEATURES: Merge with HDB Property Information
# ============================================================

# Merge the datasets on address_norm
# We only need 'year_completed' from the HDB info dataset
final_df = pd.merge(
    final_df,
    hdb_info_df[['address_norm', 'year_completed']].drop_duplicates('address_norm'),
    on='address_norm',
    how='left'
)

# Calculate lease features
CURRENT_YEAR = datetime.now().year

final_df['lease_commence_date'] = final_df['year_completed']
final_df['flat_age_at_sale'] = CURRENT_YEAR - final_df['lease_commence_date']
final_df['remaining_lease'] = 99 - final_df['flat_age_at_sale']
final_df['sold_year'] = CURRENT_YEAR

# Check how many addresses matched successfully
matched = final_df['lease_commence_date'].notna().sum()
total = len(final_df)
print(f"Successfully matched: {matched}/{total} listings ({matched/total*100:.1f}%)")

# Fill any unmatched addresses with town-level median (fallback)
for col in ['lease_commence_date', 'flat_age_at_sale', 'remaining_lease']:
    final_df[col] = final_df.groupby('hdb_town')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Drop temporary columns
#final_df = final_df.drop(columns=['address_key', 'year_completed'])

print("Lease features done.")
print(final_df[['onemap_full_address', 'lease_commence_date', 'remaining_lease', 'flat_age_at_sale']].head(10))

Successfully matched: 11951/12192 listings (98.0%)
Lease features done.
                                 onemap_full_address  lease_commence_date  \
0  903 TAMPINES AVENUE 4 TAMPINES PALMSVILLE SING...               1983.0   
1           32 BEDOK SOUTH AVENUE 2 SINGAPORE 460032               1977.0   
2         224 JURONG EAST STREET 21 SINGAPORE 600224               1984.0   
3  302 CLEMENTI AVENUE 4 CLEMENTI MEADOWS SINGAPO...               1978.0   
4   94 DAWSON ROAD SKYPARC @ DAWSON SINGAPORE 141094               2020.0   
5                   181 JELEBU ROAD SINGAPORE 670181               2001.0   
6               18 BEDOK SOUTH ROAD SINGAPORE 460018               1975.0   
7         226 JURONG EAST STREET 21 SINGAPORE 600226               1982.0   
8  706 YISHUN AVENUE 5 CHONG PANG GREEN SINGAPORE...               1983.0   
9  116A JALAN TENTERAM TENTERAM PEAK SINGAPORE 32...               2016.0   

   remaining_lease  flat_age_at_sale  
0             56.0              43.0  
1 

In [71]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12192 entries, 0 to 12191
Data columns (total 49 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        12192 non-null  str    
 1   exact_address                      12192 non-null  str    
 2   price_detail                       12192 non-null  str    
 3   town_detail                        12192 non-null  str    
 4   bedrooms_detail                    12192 non-null  float64
 5   bathrooms_detail                   11823 non-null  float64
 6   area_detail                        12192 non-null  str    
 7   floor_detail                       6082 non-null   str    
 8   property_type_detail               12192 non-null  str    
 9   address_from_url                   12192 non-null  str    
 10  listing_id                         12192 non-null  int64  
 11  postal_code                        12185 non-null  float64
 12  o

In [72]:
final_df.sample(10)

,listing_url,exact_address,price_detail,town_detail,bedrooms_detail,bathrooms_detail,area_detail,floor_detail,property_type_detail,address_from_url,...,storey_mid,flat_type,block_number,standardised_address,address_norm,year_completed,lease_commence_date,flat_age_at_sale,remaining_lease,sold_year
8734,https://www.propertyguru.com.sg/listing/hdb-fo...,703 Tampines Street 71,"S$ 888,000",Clementi,3.0,2.0,"1,281 sqft",High Floor,HDB,703 Tampines Street 71,...,16.0,4 ROOM,703,703 TAMPINES STREET 71,703 Tampines Street 71,1995.0,1995.0,31.0,68.0,2026
4361,https://www.propertyguru.com.sg/listing/hdb-fo...,430B Yishun Avenue 11,"S$ 580,000",Clementi,3.0,2.0,990 sqft,10th floor,HDB,430B Yishun Avenue 11,...,9.0,4 ROOM,430B,430B YISHUN AVENUE 11,430B Yishun Avenue 11,2014.0,2014.0,12.0,87.0,2026
8543,https://www.propertyguru.com.sg/listing/hdb-fo...,446 Hougang Avenue 8,"S$ 980,000",Clementi,3.0,3.0,"1,614 sqft",NaN,HDB,446 Hougang Avenue 8,...,16.0,4 ROOM,446,446 HOUGANG AVENUE 8,446 Hougang Avenue 8,1990.0,1990.0,36.0,63.0,2026
11822,https://www.propertyguru.com.sg/listing/hdb-fo...,114 Lengkong Tiga,"S$ 790,000",Bedok,3.0,2.0,"1,120 sqft",NaN,HDB,114 Lengkong Tiga,...,16.0,4 ROOM,114,114 LENGKONG TIGA,114 Lengkong Tiga,1988.0,1988.0,38.0,61.0,2026
7481,https://www.propertyguru.com.sg/listing/hdb-fo...,587 Cheng San Court,"S$ 420,000",Ang Mo Kio,2.0,2.0,721 sqft,NaN,HDB,587 Cheng San Court,...,16.0,3 ROOM,587,587 ANG MO KIO AVENUE 3,587 Ang Mo Kio Avenue 3,1978.0,1978.0,48.0,51.0,2026
3882,https://www.propertyguru.com.sg/listing/hdb-fo...,412 Pasir Ris Drive 6,"S$ 670,000",Clementi,3.0,2.0,"1,141 sqft",NaN,HDB,412 Pasir Ris Drive 6,...,16.0,4 ROOM,412,412 PASIR RIS DRIVE 6,412 Pasir Ris Drive 6,1989.0,1989.0,37.0,62.0,2026
4622,https://www.propertyguru.com.sg/listing/hdb-fo...,440 Ang Mo Kio Avenue 10,"S$ 486,000",Ang Mo Kio,2.0,2.0,883 sqft,High Floor,HDB,440 Ang Mo Kio Avenue 10,...,16.0,3 ROOM,440,440 ANG MO KIO AVENUE 10,440 Ang Mo Kio Avenue 10,1978.0,1978.0,48.0,51.0,2026
5132,https://www.propertyguru.com.sg/listing/hdb-fo...,263 Waterloo Street,"S$ 632,000",Central Area,2.0,1.0,731 sqft,NaN,HDB,263 Waterloo Street,...,16.0,3 ROOM,263,263 WATERLOO STREET,263 Waterloo Street,1978.0,1978.0,48.0,51.0,2026
10455,https://www.propertyguru.com.sg/listing/hdb-fo...,443B Fernvale Road,"S$ 380,000",Clementi,3.0,2.0,506 sqft,NaN,HDB,443B Fernvale Road,...,16.0,4 ROOM,443B,443B FERNVALE ROAD,443B Fernvale Road,2010.0,2010.0,16.0,83.0,2026
7920,https://www.propertyguru.com.sg/listing/hdb-fo...,125 Lorong 1 Toa Payoh,"S$ 380,000",Clementi,2.0,1.0,721 sqft,NaN,HDB,125 Lorong 1 Toa Payoh,...,16.0,3 ROOM,125,125 LORONG 1 TOA PAYOH,125 Lorong 1 Toa Payoh,1969.0,1969.0,57.0,42.0,2026


In [81]:
final_df.columns.tolist()

['listing_url',
 'onemap_full_address',
 'postal_code',
 'price_detail',
 'nearest_mrt_distance_m',
 'nearest_clinic_distance_m',
 'nearest_park_distance_m',
 'nearest_community_club_distance_m',
 'nearest_hawker_distance_m',
 'num_mrt_within_1000m',
 'num_clinic_within_1000m',
 'num_park_within_1000m',
 'num_community_club_within_1000m',
 'num_hawker_within_1000m',
 'num_amenities_within_1000m',
 'nearest_mrt_name',
 'nearest_community_club_name']

In [ ]:
# ============================================================
# Drop Unnecessary Columns
# ============================================================
# Keep only columns needed for prediction, SAI scoring,
# and final output display
cols_to_keep = [
    # Identifiers and display
    'listing_url', 'onemap_full_address', 'postal_code',
    'price_detail', 'hdb_town', 'room_count',

    # Model features
    'floor_area_sqm', 'flat_type', 'lease_commence_date',
    'remaining_lease', 'flat_age_at_sale', 'sold_year',
    'storey_mid', 'floor_category', 'region', 'is_mature_estate',
    'nearest_mrt_distance_m', 'nearest_clinic_distance_m',
    'nearest_park_distance_m', 'nearest_community_club_distance_m',
    'nearest_hawker_distance_m', 'num_mrt_within_1000m',
    'num_clinic_within_1000m', 'num_park_within_1000m',
    'num_community_club_within_1000m', 'num_hawker_within_1000m',
    'num_amenities_within_1000m', 

    # SAI display columns
    'nearest_mrt_name', 'nearest_community_club_name',
]

# Only keep columns that actually exist in the dataframe
cols_to_keep = [c for c in cols_to_keep if c in df.columns]
final_df = df[cols_to_keep].copy()

print(f"Final shape: {final_df.shape}")
print(f"Columns kept: {final_df.columns.tolist()}")


Final shape: (14771, 17)
Columns kept: ['listing_url', 'onemap_full_address', 'postal_code', 'price_detail', 'nearest_mrt_distance_m', 'nearest_clinic_distance_m', 'nearest_park_distance_m', 'nearest_community_club_distance_m', 'nearest_hawker_distance_m', 'num_mrt_within_1000m', 'num_clinic_within_1000m', 'num_park_within_1000m', 'num_community_club_within_1000m', 'num_hawker_within_1000m', 'num_amenities_within_1000m', 'nearest_mrt_name', 'nearest_community_club_name']


In [74]:
final_df.sample(10)

,listing_url,onemap_full_address,postal_code,price_detail,nearest_mrt_distance_m,nearest_clinic_distance_m,nearest_park_distance_m,nearest_community_club_distance_m,nearest_hawker_distance_m,num_mrt_within_1000m,num_clinic_within_1000m,num_park_within_1000m,num_community_club_within_1000m,num_hawker_within_1000m,num_amenities_within_1000m,nearest_mrt_name,nearest_community_club_name
11331,https://www.propertyguru.com.sg/listing/hdb-fo...,264 WATERLOO STREET SINGAPORE 180264,180264.0,"S$ 668,000",161.7,272.3,452.9,1326.8,308.3,9.0,25.0,12.0,0.0,2.0,39.0,BENCOOLEN MRT STATION,Kampong Glam CC
4230,https://www.propertyguru.com.sg/listing/hdb-fo...,70 REDHILL CLOSE REDHILL RISE SINGAPORE 150070,150070.0,"S$ 899,999",294.2,181.4,483.9,189.5,190.4,1.0,19.0,3.0,3.0,5.0,30.0,REDHILL MRT STATION,Bukit Merah CC
15625,https://www.propertyguru.com.sg/listing/hdb-fo...,44 BEDOK SOUTH ROAD BEDOK SOUTH PARKVIEW SINGA...,460044.0,"S$ 488,000",783.1,290.8,163.1,193.5,346.6,2.0,10.0,12.0,3.0,2.0,27.0,TANAH MERAH MRT STATION,Siglap CC
19143,https://www.propertyguru.com.sg/listing/hdb-fo...,NaN,NaN,"S$ 858,888",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2270,https://www.propertyguru.com.sg/listing/hdb-fo...,137 RIVERVALE STREET RIVERVALE PLAINS SINGAPOR...,540137.0,"S$ 785,000",123.1,378.0,266.4,517.8,3921.5,6.0,11.0,1.0,1.0,0.0,13.0,BAKAU LRT STATION,Rivervale CC (U/C)
2264,https://www.propertyguru.com.sg/listing/hdb-fo...,120B EDGEDALE PLAINS PUNGGOL EDGE SINGAPORE 82...,822120.0,"S$ 648,000",255.3,212.9,528.3,668.3,4622.8,8.0,13.0,1.0,2.0,0.0,16.0,MERIDIAN LRT STATION,Punggol 21 CC
5735,https://www.propertyguru.com.sg/listing/hdb-fo...,469A ADMIRALTY DRIVE BLUE RIVERVIEW SINGAPORE ...,751469.0,"S$ 550,000",682.4,399.7,569.2,841.0,2628.4,1.0,14.0,2.0,1.0,0.0,17.0,SEMBAWANG MRT STATION,Canberra CC
7531,https://www.propertyguru.com.sg/listing/for-sa...,NaN,NaN,"S$ 2,865,000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8402,https://www.propertyguru.com.sg/listing/hdb-fo...,50 LORONG 5 TOA PAYOH EAST PAYOH PALM SINGAPOR...,310050.0,"S$ 950,000",712.3,149.1,749.4,237.0,111.1,2.0,35.0,1.0,4.0,6.0,46.0,BRADDELL MRT STATION,Toa Payoh East CC (Pending relocation)
20110,https://www.propertyguru.com.sg/listing/hdb-fo...,463 JURONG WEST STREET 41 HONG KAH VILLE SINGA...,640463.0,"S$ 520,000",714.2,0.0,631.5,233.8,543.5,1.0,22.0,1.0,2.0,1.0,26.0,LAKESIDE MRT STATION,Jurong Green CC


In [75]:
# Amenity saturation thresholds (max counts)

# These values cap the maximum number of amenities counted towards the SAI score.
# This serves two main purposes:
# 1. Diminishing Returns: Having 3 MRTs nearby is excellent, but a 4th doesn't add much practical value.
# 2. Score Normalization: It prevents a location with an extreme abundance of one amenity 
#    (e.g., 20 clinics in a commercial hub) from disproportionately skewing the final SAI score.

max_counts = {
    'clinic': 10,
    'hawker': 5,
    'park': 3,
    'mrt': 3
}

In [76]:
import math
import pandas as pd

def calculate_sai_for_row(row, sliders, half_life_distance=500):
    """
    Calculates the SAI using a lenient Exponential Decay for distance for a single DataFrame row.
    Slider weights for 'clinic', 'hawker', 'park', 'mrt' should be between 1 and 10.
    The final SAI score is guaranteed to be between 0 and 100.
    """
    decay_rate = math.log(2) / half_life_distance
    
    # Extract distances from the row (default to 500m if missing or NaN)
    distances = {
        'clinic': row.get('nearest_clinic_distance_m', 500),
        'hawker': row.get('nearest_hawker_distance_m', 500),
        'park': row.get('nearest_park_distance_m', 500),
        'mrt': row.get('nearest_mrt_distance_m', 500)
    }
    
    # Extract counts from the row (default to 1 if missing or NaN)
    counts = {
        'clinic': row.get('num_clinic_within_1000m', 1),
        'hawker': row.get('num_hawker_within_1000m', 1),
        'park': row.get('num_park_within_1000m', 1),
        'mrt': row.get('num_mrt_within_1000m', 1) 
    }
    
    weighted_sum = 0
    total_weights = 0
    
    for category in ['clinic', 'hawker', 'park', 'mrt']:
        max_c = max_counts.get(category, 1)
        
        # Handle potential NaN values safely and ensure no negative values
        dist = distances[category] if pd.notna(distances[category]) else 500
        dist = max(0, dist) # Prevent negative distances from inflating score > 50
        
        count = counts[category] if pd.notna(counts[category]) else 1
        count = max(0, count) # Prevent negative counts
        
        # Get the slider weight (expecting 1-10, default to 1 if missing)
        weight = sliders.get(category, 1)
        
        # Cap count to max_c to ensure count_score doesn't exceed 50
        count_capped = min(count, max_c)
        
        # 1. Lenient Exponential Decay for Distance (Max 50 points)
        dist_score = 50 * math.exp(-decay_rate * dist)
        
        # 2. Linear scale for Density (Max 50 points)
        count_score = 50 * (count_capped / max_c) if max_c > 0 else 0
        
        # Total Category Score (Max 100 points)
        score = dist_score + count_score
        
        # Applying the individual slider weight to the category score
        weighted_sum += (score * weight)
        # Keeping track of the total slider values used
        total_weights += weight
        
    if total_weights == 0:
        return 0.0

    # Normalization of SAI score     
    final_sai = weighted_sum / total_weights
    
    # Strictly bound the final score between 0 and 100
    final_sai = max(0.0, min(100.0, final_sai))
    
    return round(final_sai, 1)

In [79]:
# ==========================================
# DATAFRAME APPLICATION EXAMPLE
# ==========================================

# 2. Define user sliders mapped to the new categories
# user_sliders = {
#     'clinic': 7, 
#     'hawker': 8, 
#     'park': 4, 
#     'mrt': 9
# }

import re 

def final_output(df_input, town, min_rooms, user_sliders, budget=None):
    """
    Filters the dataframe based on town, minimum room count, and an optional budget.
    """
    df = df_input.copy()

    # Convert to nullable integer ('Int64') to safely drop the .0 while ignoring NaNs
    # Convert to string
    df['postal_code'] = df['postal_code'].astype('Int64').astype(str)

    # When pandas converts missing 'Int64' values to strings, it writes "<NA>". 
    # Replace those with empty strings.
    df['postal_code'] = df['postal_code'].replace('<NA>', '')

    clean_numeric_price = (
    df['price_detail']
      .replace(r'\D', '', regex=True)
      .fillna('0')             # treat missing as "0"
      .astype(int)
)
    df['formatted_price'] = clean_numeric_price.map('${:,.0f}'.format)

    # 1. Filter by town (making it case-insensitive for robustness)
    filtered_df = df[df['hdb_town'].str.lower() == town.lower()]
    
    # 2. Filter by minimum number of rooms
    filtered_df = filtered_df[filtered_df['room_count'] >= min_rooms]
    
    # Filter by budget if it is provided
    filtered_df['price_detail'] = filtered_df['price_detail'].replace(r'\D', '', regex=True).astype(int)
    if budget is not None:
        filtered_df = filtered_df[filtered_df['price_detail'] <= budget]
    
    # First sort by onemap_full_address, then by price_detail (ascending)
    filtered_df = filtered_df.sort_values(by=["onemap_full_address", "price_detail"], ascending=[True, True])

    # Then drop duplicates on onemap_full_address, keeping the first (lowest price_detail)
    filtered_df = filtered_df.drop_duplicates(subset=["onemap_full_address"], keep="first")
        
    # 3. Apply the function across the dataframe (axis=1 means row-by-row)
    filtered_df['SAI_Score'] = filtered_df.apply(lambda row: calculate_sai_for_row(row, user_sliders), axis=1)

    # 4. Rank by SAI_Score in descending order
    ranked_df = filtered_df.sort_values(by=["SAI_Score", "price_detail"], 
                           ascending=[False, True])
    
    # 5. Get the top 3 rows
    top_3_df = ranked_df.head(3)
    
    # 6. Select and return only the requested columns
    columns_to_return = ['listing_url', 'onemap_full_address', 'hdb_town', 'postal_code', 
                         'formatted_price', 'room_count', 'SAI_Score',
                         'nearest_mrt_name', 'nearest_mrt_distance_m', 'num_mrt_within_1000m', 
                         'nearest_clinic_distance_m', 'num_clinic_within_1000m', 
                         'nearest_community_club_name', 'nearest_community_club_distance_m', 
                         'nearest_hawker_distance_m', 'num_hawker_within_1000m', 
                         'num_park_within_1000m', 'num_park_within_1000m'
]
    
    
    return top_3_df[columns_to_return]

In [80]:
# Test Case 1: Filter by town and min_rooms only (no budget)
user_sliders = {
    'clinic': 8, 
    'hawker': 5, 
    'park': 7, 
    'mrt': 10, 
}

print("Test Case 1: Tampines, Min 3 rooms, No budget limit")
result_1 = final_output(final_df, town='Tampines', min_rooms=3, user_sliders=user_sliders)
result_1.T

Test Case 1: Tampines, Min 3 rooms, No budget limit


KeyError: 'hdb_town'

In [ ]:
# Test Case 2: Filter by town, min_rooms, and budget
print("Test Case 2: Tampines, Min 3 rooms, Budget <= 600,000")
result_2 = final_output(final_df, town='Tampines', min_rooms=3, budget=600000, user_sliders=user_sliders)
result_2.T

Test Case 2: Tampines, Min 3 rooms, Budget <= 600,000


,18073,2103,1934
listing_url,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...
onemap_full_address,406 TAMPINES STREET 41 SUN PLAZA GREEN SINGAPO...,244 TAMPINES STREET 21 SINGAPORE 521244,419 TAMPINES STREET 41 SUN PLAZA GARDENS SINGA...
hdb_town,Tampines,Tampines,Tampines
postal_code,520406,521244,520419
formatted_price,"$488,000","$487,000","$599,000"
room_count,3.0,3.0,4.0
SAI_Score,69.8,68.2,67.9
nearest_mrt_name,TAMPINES MRT STATION,TAMPINES MRT STATION,TAMPINES MRT STATION
nearest_mrt_distance_m,403.0,227.4,471.6
num_mrt_within_1000m,2.0,2.0,2.0


In [ ]:
# Test Case 3: Filter by town, min_rooms, and budget
print("Test Case 3: Jurong East, Min 3 rooms, Budget <= 400,000")
result_3 = final_output(final_df, town='Jurong East', min_rooms=3, budget=400000, user_sliders=user_sliders)
result_3.T

Test Case 3: Jurong East, Min 3 rooms, Budget <= 400,000


,4035,2709,2394
listing_url,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...,https://www.propertyguru.com.sg/listing/hdb-fo...
onemap_full_address,247 JURONG EAST STREET 24 SINGAPORE 600247,110 JURONG EAST STREET 13 HAPPY TALENT CHILDCA...,303 JURONG EAST STREET 32 HONG KAH EAST GARDEN...
hdb_town,Jurong East,Jurong East,Jurong East
postal_code,600247,600110,600303
formatted_price,"$388,000","$400,000","$398,000"
room_count,3.0,3.0,3.0
SAI_Score,69.0,68.2,68.1
nearest_mrt_name,CHINESE GARDEN MRT STATION,CHINESE GARDEN MRT STATION,CHINESE GARDEN MRT STATION
nearest_mrt_distance_m,627.1,433.2,440.3
num_mrt_within_1000m,2.0,2.0,1.0
